# Linked Entity To Human

This notebook creates a human-readable linked-entity export from the canonical `motel-db/linked_entity/linked_entity.yaml` file.

The main conversion rule is to expand foreign keys into `ID (Name)`, for example `ATTR_00002` becomes `ATTR_00002 (Technology Maturity)`.


## What It Expands

The notebook resolves names from the MOTEL registries for:

- `tech_id`
- `process_id`
- `source_id`
- `carrier_id`
- `attribute_id`
- scope references such as `geographic_scope`, `temporal_scope`, `capacity_scope`, and `system_boundary`

Unregistered tokens such as `[unregistered: ratios_in]` are preserved as they are.


In [ ]:
from __future__ import annotations

import csv
from copy import deepcopy
from pathlib import Path

import yaml


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "motel-db").is_dir() and (candidate / "schema_human").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root. Start the notebook from inside motel-platform.")


PROJECT_ROOT = find_project_root()
DB_DIR = PROJECT_ROOT / "motel-db"
INPUT_PATH = DB_DIR / "linked_entity" / "linked_entity.yaml"
OUTPUT_DIR = DB_DIR / "linked_entity_human"
OUTPUT_PATH = OUTPUT_DIR / "linked_entity.yaml"

print(f"Project root: {PROJECT_ROOT}")
print(f"Canonical linked entities: {INPUT_PATH}")
print(f"Human-readable output target: {OUTPUT_PATH}")


In [ ]:
LOOKUP_SPECS = {
    "tech_id": {
        "path": DB_DIR / "secondary" / "technology.csv",
        "id_field": "tech_id",
        "name_field": "technology_name",
    },
    "process_id": {
        "path": DB_DIR / "secondary" / "process.csv",
        "id_field": "process_id",
        "name_field": "process_name",
    },
    "source_id": {
        "path": DB_DIR / "secondary" / "source.csv",
        "id_field": "source_id",
        "name_field": "source_name",
    },
    "carrier_id": {
        "path": DB_DIR / "controlled_vocabulary" / "carrier.csv",
        "id_field": "carrier_id",
        "name_field": "carrier_name",
    },
    "attribute_id": {
        "path": DB_DIR / "controlled_vocabulary" / "attribute.csv",
        "id_field": "attribute_id",
        "name_field": "attribute_name",
    },
    "geographic_scope": {
        "path": DB_DIR / "controlled_vocabulary" / "geographic_scope.csv",
        "id_field": "geographic_scope",
        "name_field": "geographic_scope_description",
    },
    "temporal_scope": {
        "path": DB_DIR / "controlled_vocabulary" / "temporal_scope.csv",
        "id_field": "temporal_scope",
        "name_field": "temporal_scope_description",
    },
    "capacity_scope": {
        "path": DB_DIR / "controlled_vocabulary" / "capacity_scope.csv",
        "id_field": "capacity_scope",
        "name_field": "capacity_scope_description",
    },
    "system_boundary": {
        "path": DB_DIR / "controlled_vocabulary" / "system_boundary.csv",
        "id_field": "system_boundary",
        "name_field": "system_boundary_description",
    },
}


def load_lookup(path: Path, id_field: str, name_field: str) -> dict[str, str]:
    with open(path, encoding="utf-8", newline="") as handle:
        rows = csv.DictReader(handle)
        return {
            str(row[id_field]).strip(): str(row.get(name_field, "")).strip()
            for row in rows
            if str(row.get(id_field, "")).strip()
        }


LOOKUPS = {
    key: load_lookup(spec["path"], spec["id_field"], spec["name_field"])
    for key, spec in LOOKUP_SPECS.items()
}

for key, lookup in LOOKUPS.items():
    print(f"{key}: {len(lookup)} labels loaded")


In [ ]:
def format_ref(ref_id: str, lookup: dict[str, str]) -> str:
    ref_id = str(ref_id or "").strip()
    if not ref_id:
        return ref_id
    if ref_id.startswith("[") and ref_id.endswith("]"):
        return ref_id
    label = lookup.get(ref_id, "").strip()
    return f"{ref_id} ({label})" if label else ref_id


def enrich_flow_entries(entries: list[dict], carrier_lookup: dict[str, str]) -> list[dict]:
    enriched = []
    for entry in entries or []:
        item = deepcopy(entry)
        if "carrier_id" in item:
            item["carrier_id"] = format_ref(item["carrier_id"], carrier_lookup)
        enriched.append(item)
    return enriched


def enrich_sources(sources: list[dict], source_lookup: dict[str, str], attribute_lookup: dict[str, str]) -> list[dict]:
    enriched = []
    for source in sources or []:
        item = deepcopy(source)
        if "source_id" in item:
            item["source_id"] = format_ref(item["source_id"], source_lookup)
        item["linked_attribute_ids"] = [
            format_ref(attribute_id, attribute_lookup)
            for attribute_id in item.get("linked_attribute_ids", [])
        ]
        enriched.append(item)
    return enriched


def enrich_values(values: list[dict], attribute_lookup: dict[str, str]) -> list[dict]:
    enriched = []
    for value in values or []:
        item = deepcopy(value)
        if "attribute_id" in item:
            item["attribute_id"] = format_ref(item["attribute_id"], attribute_lookup)
        enriched.append(item)
    return enriched


def enrich_scope(scope: dict, lookups: dict[str, dict[str, str]]) -> dict:
    item = deepcopy(scope or {})
    for key in ("geographic_scope", "temporal_scope", "capacity_scope", "system_boundary"):
        if key in item:
            item[key] = format_ref(item[key], lookups[key])
    return item


def enrich_entity(entity: dict, lookups: dict[str, dict[str, str]]) -> dict:
    item = deepcopy(entity)
    item["tech_id"] = format_ref(item.get("tech_id", ""), lookups["tech_id"])
    item["process_id"] = format_ref(item.get("process_id", ""), lookups["process_id"])
    item["scope"] = enrich_scope(item.get("scope", {}), lookups)

    balancing = deepcopy(item.get("balancing", {}))
    balancing["inputs"] = enrich_flow_entries(balancing.get("inputs", []), lookups["carrier_id"])
    balancing["outputs"] = enrich_flow_entries(balancing.get("outputs", []), lookups["carrier_id"])
    if "main_input" in balancing:
        balancing["main_input"] = format_ref(balancing.get("main_input", ""), lookups["carrier_id"])
    if "main_output" in balancing:
        balancing["main_output"] = format_ref(balancing.get("main_output", ""), lookups["carrier_id"])
    item["balancing"] = balancing

    item["sources"] = enrich_sources(item.get("sources", []), lookups["source_id"], lookups["attribute_id"])
    item["values"] = enrich_values(item.get("values", []), lookups["attribute_id"])
    return item


In [ ]:
with open(INPUT_PATH, encoding="utf-8") as handle:
    linked_entities = yaml.safe_load(handle) or []

linked_entities_human = [enrich_entity(entity, LOOKUPS) for entity in linked_entities]

print(f"Linked entities loaded: {len(linked_entities)}")
linked_entities_human[:2]


In [ ]:
# Run this cell only when you want to materialize the human-readable export.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as handle:
    yaml.safe_dump(
        linked_entities_human,
        handle,
        allow_unicode=True,
        sort_keys=False,
        default_flow_style=False,
    )

print(f"Wrote {len(linked_entities_human)} human-readable linked entities -> {OUTPUT_PATH}")
